## Annotation automatique des images

Ce notebook permet de générer automatiquement des annotations pour chaque image du dossier `images/`. Il remplit deux objectifs principaux :

1. **Extraction des couleurs dominantes**  
   - Chaque image est convertie en vecteurs RGB.
   - Un algorithme de clustering (`KMeans`) est utilisé pour identifier les couleurs dominantes.
   - Les couleurs sont codées en hexadécimal (ex: `#aabbcc`).

2. **Extraction des métadonnées EXIF**  
   - Format de l’image, dimensions, mode colorimétrique.
   - Appareil photo utilisé, date de capture, ISO, exposition, focale.
   - Coordonnées GPS (si disponibles), converties en latitude / longitude.

Toutes ces informations sont combinées dans un fichier `annotations.json`, structuré par image. Ce fichier servira de base pour les étapes suivantes : analyse, visualisation, recommandation.

Le script ignore automatiquement les images :
- Trop volumineuses (> 50 Mo)
- Ne contenant pas de données exploitables
- Avec des erreurs de lecture ou d’EXIF

Ce traitement est entièrement automatisé.


In [ ]:
import os
import json
import cv2
import numpy as np
from PIL import Image, ExifTags
from sklearn.cluster import KMeans

IMAGE_FOLDER = "../images"
ANNOTATION_FILE = "JSON/annotations.json"

def convert_to_serializable(obj):
    try:
        return float(obj)
    except (TypeError, ValueError):
        return str(obj)

def safe_div(num, denom):
    try:
        return float(num) / float(denom) if denom != 0 else 0
    except:
        return 0

def get_exif_data(image):
    try:
        exif_data = image._getexif()
        if not exif_data:
            return {}
        return {
            ExifTags.TAGS.get(tag, tag): convert_to_serializable(value)
            for tag, value in exif_data.items()
        }
    except AttributeError:
        return {}

def get_gps_info(exif_data):
    if "GPSInfo" not in exif_data:
        return None
    gps_info = exif_data["GPSInfo"]
    if 2 not in gps_info or 4 not in gps_info:
        return None
    try:
        lat = [safe_div(x[0], x[1]) if isinstance(x, tuple) else float(x) for x in gps_info[2]]
        lon = [safe_div(x[0], x[1]) if isinstance(x, tuple) else float(x) for x in gps_info[4]]
        latitude = lat[0] + (lat[1] / 60.0) + (lat[2] / 3600.0)
        longitude = lon[0] + (lon[1] / 60.0) + (lon[2] / 3600.0)
        if gps_info.get(1) == 'S':
            latitude = -latitude
        if gps_info.get(3) == 'W':
            longitude = -longitude
        return {"latitude": latitude, "longitude": longitude}
    except Exception as e:
        print(f"Erreur GPS : {e}")
        return None

def get_dominant_colors(image_path, num_colors=3):
    img = cv2.imread(image_path)
    if img is None:
        return []
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (200, 200))
    pixels = img.reshape(-1, 3)
    kmeans = KMeans(n_clusters=num_colors, random_state=42).fit(pixels)
    return ["#{:02x}{:02x}{:02x}".format(int(r), int(g), int(b)) for r, g, b in kmeans.cluster_centers_]

def get_image_metadata(image_path):
    with Image.open(image_path) as img:
        exif_data = get_exif_data(img)
        gps_info = get_gps_info(exif_data)
        return {
            "format": img.format,
            "size": img.size,
            "mode": img.mode,
            "device": exif_data.get("Make", "Unknown") + " " + exif_data.get("Model", "Unknown"),
            "capture_date": exif_data.get("DateTimeOriginal", "Unknown"),
            "iso": exif_data.get("ISOSpeedRatings", "Unknown"),
            "focal_length": exif_data.get("FocalLength", "Unknown"),
            "exposure_time": exif_data.get("ExposureTime", "Unknown"),
            "gps": gps_info if gps_info else "No GPS Data"
        }

def process_images():
    annotations = {}
    for filename in os.listdir(IMAGE_FOLDER):
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        full_path = os.path.join(IMAGE_FOLDER, filename)

        # Skip les très grosses images
        if os.path.getsize(full_path) > 50 * 1024 * 1024:
            print(f"Image trop lourde : {filename}, ignorée.")
            continue

        try:
            dominant_colors = get_dominant_colors(full_path)
            metadata = get_image_metadata(full_path)
            annotations[filename] = {
                "dominant_colors": dominant_colors,
                "metadata": metadata
            }
        except Exception as e:
            print(f"Erreur sur {filename} : {e}")
            continue

    with open(ANNOTATION_FILE, "w", encoding="utf-8") as f:
        json.dump(annotations, f, indent=4, ensure_ascii=False)

# Exécute tout
process_images()


Erreur sur %E9%B8%9F%E7%9E%B0%E5%A4%A9%E6%B0%B4%20-%20panoramio.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Pedestrian%20road%20with%20pavements%2C%20paper%20umbrellas%20and%20people%20in%20yukata%2C%20Higashiyama-ku%2C%20Kyoto%2C%20Japan.jpg : 'in <string>' requires string as left operand, not int
Erreur sur C%C3%A0%20Mau%20city%20center.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Downtown%20Provo.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Witten%2020090218.jpg : 'in <string>' requires string as left operand, not int


libpng warning: iCCP: known incorrect sRGB profile


Erreur sur 20130610%20-%20Monumento%20aos%20Her%C3%B3is%20da%20Independ%C3%AAncia.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Fontana%20California%20Overview.JPG : division by zero
Erreur sur Kailasanathar%20temple%20Kanchipuram%20%2816%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Nyerere%20Square%20in%20Dodoma.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Mysore%20Palace%2C%20India%20%28photo%20-%20Jim%20Ankan%20Deka%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur The%20west%20panorama%20of%20Shenzhen2021.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Leon%20Cathedral%202018.jpg : 'in <string>' requires string as left operand, not int
Erreur sur 2017-05-27%20Plac%20Zamkowy%20w%20Warszawie%201.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Baqtoeste.jpg : division by zero
Erreur sur L%27Hospitalet%20de%20Llobregat.jpg : 'in <st

libpng warning: iCCP: known incorrect sRGB profile


Erreur sur Hachioji%20%2827980514509%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Aerial%20photograph%20of%20Kananga.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Jos%2C%20looking%20south%20from%20Anglo-Jos.jpg : 'in <string>' requires string as left operand, not int
Erreur sur PL%20Rzesz%C3%B3w%2C%20ratusz%202021-05-04--11-00-20%20%28cropped%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Skyline%20over%20the%20Hong%20Kong%20Harbour.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Panoramic%20view%20from%20Juche%20Tower.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Illinois%20State%20Capitol%20distance.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Pet%C5%99%C3%ADn%20Tower%20View%20IMG%203020.JPG : 'in <string>' requires string as left operand, not int
Erreur sur Arial%20view%20of%20rajshahi.jpg : 'in <string>' requires string as 

...

Erreur sur Theodor-Heuss-Br%C3%BCcke%2C%201902231957%2C%20ako.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Castello%20Aragonese%28Taranto%29.JPG : 'in <string>' requires string as left operand, not int
Erreur sur Nishijuku%20Hanadach%C5%8D%2C%20Toyohashi-shi%2C%20Aichi-ken%20441-8019%2C%20Japan%20-%20panoramio.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Luftbild%20vom%20Diokletianpalast%20in%20Split%2C%20Kroatien%20%2848608754492%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Hamm%2C%20Germany%20-%20panoramio%20%281459%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Tianhan%20bridge%20in%20Hanzhong2021.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Lakhota%20Tower%20in%20evening%20by%20Rangilo.JPG : 'in <string>' requires string as left operand, not int
Erreur sur Skyline%20of%20Xianyang%20%28in%20sandstorm%29%2C%20April%2028%202023.jpg : 'in <string>' 

/Users/maxime/Documents/Git Projects/TraitementDonneeMassiveProjet/venv/lib/python3.13/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (99998272 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
libpng warning: iCCP: known incorrect sRGB profile


Erreur sur Nicaragua%202017-03-15%20%2833978834325%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Siegen%20-%20panoramio%20%289%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur 00%20sea%20towers%20%28April%202018%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Eye%20Of%20Lahore%20%28Minar%20e%20Pakistan%29%20evening.jpg : 'in <string>' requires string as left operand, not int


libpng warning: iCCP: known incorrect sRGB profile


Erreur sur Rishon%20LeZion%20West%20Aerial%20View.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Yantai2017.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Panor%C3%A1mica%20Guadalajara%20desde%20edificio%20Bansi%20hacia%20norte%20%28cropped%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Havenwelten%20%C3%9Cberblick%20Bremerhaven%202013.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Rome%20Skyline%20%288012016319%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Kasugai%20City%20Hall%20ac%20%281%29.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Buenos%20Aires%20-%20Puerto%20Madero.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Karlsruhe%20asv2022-10%20img07%20Schloss%20Karlsruhe.jpg : 'in <string>' requires string as left operand, not int
Erreur sur Toyohashi%20City%20Hall%20%282011.09.15%29.jpg : division by ze